# Dark & Radiation Background Removal


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import matplotlib.dates as mdates

from avg import load_and_filter_data
'''
s4 = np.datetime64("2026-02-07T00:00:00")
e4 = np.datetime64("2026-03-02T00:00:00")
'''
# ------------------------ PARAMETERS ------------------------
start_datetime_str = "2026-03-01T00:00:00"
end_datetime_str = "2027-03-02T00:00:00"
filter_neg = True   # Set to True to filter out negative FOV values, False to keep all values
beta_max = 0.5     # Set to a value less than 360 to filter out data points with beta angles above the specified maximum, or set to 360 to keep all values
# ------------------------ PARAMETERS ------------------------

# Load and filter data
wfi_avg_fov_mean, nfi_avg_fov_mean, wfi_avg_fov_mean_uncorrected, nfi_avg_fov_mean_uncorrected, wfi_time, nfi_time, wfi_roll_angles, wfi_beta_angles, nfi_roll_angles, nfi_beta_angles = load_and_filter_data(
    start_datetime_str=start_datetime_str,
    end_datetime_str=end_datetime_str,
    filter_neg=filter_neg,
    beta_max=beta_max)

In [32]:
"""
Regression Analysis: wfi_mean ~ nfi_mean + roll_angle + beta_angle

- roll_angle and beta_angle are in degrees (converted to radians for trig features)
- Outputs: summary table, coefficients, diagnostics, and residual plots
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from scipy import stats

# Create dataframe from existing data

"""
Regression Analysis: wfi_mean ~ nfi_mean + roll_angle + beta_angle

- roll_angle and beta_angle are in degrees (converted to radians for trig features)
- Outputs: summary table, coefficients, diagnostics, and residual plots
"""
def to_numeric_time(t):
    t = np.asarray(t)
    if np.issubdtype(t.dtype, np.datetime64):
        # Convert to float (nanoseconds since epoch), then to seconds
        return t.astype('datetime64[ns]').astype(np.float64)
    return t.astype(np.float64)


# ── 1. Interpolate nfi_mean onto wfi_mean time grid ──────────────────────────
# Assumed pre-loaded variables:
#   wfi_mean   : array-like, shape (N,)  — WFI flux values
#   wfi_time   : array-like, shape (N,)  — observation times for wfi_mean
#   nfi_mean   : array-like, shape (M,)  — NFI flux values  (M ≠ N in general)
#   nfi_time   : array-like, shape (M,)  — observation times for nfi_mean
#   roll_angle : array-like, shape (N,)  — roll angle in degrees
#   beta_angle : array-like, shape (N,)  — beta angle in degrees


wfi_time_arr = to_numeric_time(wfi_time)
nfi_time_arr = to_numeric_time(nfi_time)

nfi_mean_arr = nfi_avg_fov_mean
wfi_mean = wfi_avg_fov_mean - 4
roll_angle = wfi_roll_angles
beta_angle = wfi_beta_angles

# Warn if wfi_time falls outside the nfi_time range (extrapolation risk)
if wfi_time_arr.min() < nfi_time_arr.min() or wfi_time_arr.max() > nfi_time_arr.max():
    print(
        "WARNING: wfi_time extends beyond nfi_time bounds. "
        "Boundary nfi_mean values will be used for out-of-range points."
    )

# Linear interpolation — bounded at nfi_time edges to avoid extrapolation
nfi_mean_interp = np.interp(wfi_time_arr, nfi_time_arr, nfi_mean_arr)

print(f"nfi_mean interpolated: {len(nfi_mean_arr)} → {len(nfi_mean_interp)} points")

# ── 2. Assemble aligned DataFrame ────────────────────────────────────────────
df = pd.DataFrame({
    "wfi_mean"  : np.asarray(wfi_mean,   dtype=float),
    "roll_angle": np.asarray(roll_angle, dtype=float),
    "beta_angle": np.asarray(beta_angle, dtype=float),
})

# Drop rows where any value is NaN (e.g. from source data gaps)
df = df.dropna()

print(f"Rows used in analysis: {len(df)}")

# ── 2. Feature Engineering ────────────────────────────────────────────────────
# Convert angles to radians (useful for optional trig features)
df["roll_rad"]  = np.sin(np.deg2rad(df["roll_angle"]))
df["beta_rad"]  = np.cos(np.deg2rad(df["beta_angle"]))

# ── 3. Build Design Matrix ────────────────────────────────────────────────────
# Predictors: nfi_mean, roll_angle (deg), beta_angle (deg)
# Add an intercept via statsmodels convention
X = df[["roll_angle", "beta_angle"]]
X = sm.add_constant(X)   # adds column 'const'
y = df["wfi_mean"]

# ── 4. Fit OLS Model ──────────────────────────────────────────────────────────
model  = sm.OLS(y, X).fit()
print("\n" + "=" * 60)
print(model.summary())
print("=" * 60)

# ── 5. Key Metrics ────────────────────────────────────────────────────────────
print("\n── Key Metrics ──")
print(f"  R²            : {model.rsquared:.4f}")
print(f"  Adjusted R²   : {model.rsquared_adj:.4f}")
print(f"  F-statistic   : {model.fvalue:.4f}  (p={model.f_pvalue:.4e})")
print(f"  AIC           : {model.aic:.2f}")
print(f"  BIC           : {model.bic:.2f}")

# ── 6. Coefficient Table ──────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    "Coefficient": model.params,
    "Std Error"  : model.bse,
    "t-value"    : model.tvalues,
    "p-value"    : model.pvalues,
    "CI Lower"   : model.conf_int()[0],
    "CI Upper"   : model.conf_int()[1],
})
print("\n── Coefficient Table ──")
print(coef_df.round(6).to_string())

# ── 7. Variance Inflation Factors (multicollinearity) ────────────────────────
print("\n── Variance Inflation Factors ──")
vif_data = pd.DataFrame({
    "Feature": X.columns,
    "VIF"    : [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif_data.to_string(index=False))
print("  (VIF > 10 suggests problematic multicollinearity)")

# ── 8. Residual Diagnostics ───────────────────────────────────────────────────
residuals   = model.resid
fitted      = model.fittedvalues
dw_stat     = durbin_watson(residuals)
_, sw_p     = stats.shapiro(residuals)

print("\n── Residual Diagnostics ──")
print(f"  Durbin-Watson statistic : {dw_stat:.4f}  (≈2 = no autocorrelation)")
print(f"  Shapiro-Wilk p-value    : {sw_p:.4e}  ({'normal' if sw_p > 0.05 else 'non-normal'} residuals at α=0.05)")

# ── 9. Diagnostic Plots ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("OLS Regression Diagnostics\n(wfi_mean ~ nfi_mean + roll_angle + beta_angle)",
             fontsize=13, fontweight="bold")

# 9a. Residuals vs Fitted
axes[0, 0].scatter(fitted, residuals, alpha=0.5, edgecolors="k", linewidths=0.3)
axes[0, 0].axhline(0, color="red", linestyle="--")
axes[0, 0].set_xlabel("Fitted values")
axes[0, 0].set_ylabel("Residuals")
axes[0, 0].set_title("Residuals vs Fitted")

# 9b. Q-Q plot
sm.qqplot(residuals, line="s", ax=axes[0, 1], alpha=0.5)
axes[0, 1].set_title("Normal Q-Q Plot")

# 9c. Scale-Location (sqrt |residuals| vs fitted)
sqrt_abs_resid = np.sqrt(np.abs(residuals))
axes[1, 0].scatter(fitted, sqrt_abs_resid, alpha=0.5, edgecolors="k", linewidths=0.3)
axes[1, 0].set_xlabel("Fitted values")
axes[1, 0].set_ylabel("√|Residuals|")
axes[1, 0].set_title("Scale-Location")

# 9d. Actual vs Predicted
axes[1, 1].scatter(y, fitted, alpha=0.5, edgecolors="k", linewidths=0.3)
min_val = min(y.min(), fitted.min())
max_val = max(y.max(), fitted.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], "r--")
axes[1, 1].set_xlabel("Actual wfi_mean")
axes[1, 1].set_ylabel("Predicted wfi_mean")
axes[1, 1].set_title("Actual vs Predicted")

plt.tight_layout()
plt.savefig("regression_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nDiagnostic plot saved → regression_diagnostics.png")

ValueError: zero-size array to reduction operation minimum which has no identity